In [1]:
import pandas as pd
import numpy as np

In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity="all"

## 피봇테이블

- 가지고 있는 데이터원본을 원하는 형태의 가공된 정보를 보여주는 것
- 자료의 형태를 변경하기 위해 많이 사용하는 방법
- `pivot_table(data, values=None. index=None, columns=None, aggfunc='mean', margins=False, margins_name="All")`
    - `data`: 분석할 데이터 프레임, 메서드 형식일 때는 필요하지 않음
    - `values`: 분석할 데이터프레임에서 분석할 열
    - `index`: 행 인덱스로 들어갈 키열
    - `columns`: 열 인덱스로 들어갈 키열
    - `fill_value`: NaN이 표출될 때 대체값 지정
    - `margins`: 모든 데이터를 분석할 결과를 행으로 표출할 지 여부
    - `margins_name`: `margins`가 표출될 때 그 열(행)의 이름

#### 피봇테이블을 작성할 때 반드시 설정해야 하는 인수
- `data`: 사용 데이터프레임
- `index`: 행 인덱스로 사용할 필드(기준 필드)
- 인덱스 명을 제외한 나머지 값(data)은 **수치 데이터**만 사용함
- 기본 함수가 **평균함수**이기 때문에 각 데이터의 평균값이 반환

In [3]:
data = {
    '도시':['서울','서울','서울','부산','부산','부산','인천','인천'],
    '연도':['2015','2010','2005','2015','2010','2005','2015','2010'],
    '인구':[9904312, 9631482, 9762546, 3448737, 3393191, 3512547, 2890451, 263203],
    '지역':['수도권','수도권','수도권','경상권','경상권','경상권','수도권','수도권']
}
columns = ['도시', '연도', '인구', '지역']
df1=pd.DataFrame(data, columns=columns)
df1

,도시,연도,인구,지역
0,서울,2015,9904312,수도권
1,서울,2010,9631482,수도권
2,서울,2005,9762546,수도권
3,부산,2015,3448737,경상권
4,부산,2010,3393191,경상권
5,부산,2005,3512547,경상권
6,인천,2015,2890451,수도권
7,인천,2010,263203,수도권


In [4]:
# 각 도시에 대한 연도별 인구 평균
df1.pivot_table(index="도시", columns='연도', values='인구')

연도,2005,2010,2015
도시,,,
부산,3512547.0,3393191.0,3448737.0
서울,9762546.0,9631482.0,9904312.0
인천,NaN,263203.0,2890451.0


In [6]:
# 각 지역별 도시에 대한 연도별 인구 평균
df1.pivot_table(index=['지역','도시'], columns='연도', values='인구')

연도           2005       2010       2015
지역  도시                                 
경상권 부산  3512547.0  3393191.0  3448737.0
수도권 서울  9762546.0  9631482.0  9904312.0
    인천        NaN   263203.0  2890451.0

In [8]:
import seaborn as sns

df = sns.load_dataset('titanic')[['age','sex','class','fare','survived']]
df.head()

,age,sex,class,fare,survived
0,22.0,male,Third,7.2500,0
1,38.0,female,First,71.2833,1
2,26.0,female,Third,7.9250,1
3,35.0,female,First,53.1000,1
4,35.0,male,Third,8.0500,0


In [9]:
# 선실 등급별로 숙박객의 성별 평균 나이
df.pivot_table(index='class', columns='sex', values='age')

sex,female,male
class,,
First,34.611765,41.281386
Second,28.722973,30.740707
Third,21.750000,26.507589


In [10]:
# 각 선실 등급별 숙박객의 성별에 따른 생존자 수와 생존율
pdf1 = pd.pivot_table(df,
                     index='class',
                     columns='sex',
                     values='survived',
                     aggfunc=['mean', 'sum'])
pdf1

mean              sum     
sex       female      male female male
class                                 
First   0.968085  0.368852     91   45
Second  0.921053  0.157407     70   17
Third   0.500000  0.135447     72   47

In [11]:
# 각 선실 등급에 따른 성별에 대해 생존여부별로 나이와 티켓값의 평균과 최댓값을 산출
pdf2 = pd.pivot_table(df,
                     index=['class', 'sex'],
                     columns='survived',
                     values=['age', 'fare'],
                     aggfunc=['mean', 'max'])
pdf2

mean                                      max        \
                     age                   fare               age         
survived               0          1           0           1     0     1   
class  sex                                                                
First  female  25.666667  34.939024  110.604167  105.978159  50.0  63.0   
       male    44.581967  36.248000   62.894910   74.637320  71.0  80.0   
Second female  36.000000  28.080882   18.250000   22.288989  57.0  55.0   
       male    33.369048  16.022000   19.488965   21.095100  70.0  62.0   
Third  female  23.818182  19.329787   19.773093   12.464526  48.0  63.0   
       male    27.255814  22.274211   12.204469   15.579696  74.0  45.0   

                                 
                 fare            
survived            0         1  
class  sex                       
First  female  151.55  512.3292  
       male    263.00  512.3292  
Second female   26.00   65.0000  
       male     73.50   39.0000  
Third  female   69.55   31.3875  
       male     69.55   56.4958

### 그룹 분석
- 만약 키가 지정하는 조건에 맞는 데이터가 하나 이상이라서 데이터 그룹을 이루는 경우에는 그룹분석을 해야 함
- 피봇테이블과 달리 키에 의해서 결정되는 데이터가 여러개가 있을 경우 미리 지정한 연산을 통해 그 그룹 데이터의 대푯값을 계산 하는 것
- `groupby` 메서드를 사용해 그룹분석 진행

#### groupby 메서드
- 그룹 데이터를 나타내는 **GroupBy 클래스 객체** 반환

#### GroupBy 클래스 객체의 그룹연산 메서드
- `size`, `count`: 그룹 데이터의 개수
- `mean`, `median`, `min`, `max`
- `sum`, `prod`, `std`, `var`, `quantile`: 그룹 데이터의 합계, 곱, 표준편차, 분산, 사분위수
- `first`, `last`

#### 이외에도 많이 사용되는 그룹 연산
- `agg`, `aggregate`
    - 만약 원하는 그룹연산이 없는 경우 함수를 만들고 이 함수를 `agg`에 전달
    - 또는 여러가지 그룹연산을 동시에 하고 싶은 경우 함수 이름 문자열의 리스트를 전달
- `describe`
    - 하나의 그룹 대표값이 아니라 여러개의 값을 데이터프레임으로 구현
- `apply`
    - `describe` 처럼 하나의 대표값이 아닌 데이터프레임을 출력하지만 원하는 그룹연산이 없는 경우에 사용
- `transform`
    - 그룹에 대한 대표값을 만드는 것이 아니라 그룹별 계산을 통해 데이터 자체를 변형

In [12]:
np.random.seed(0)
df2 = pd.DataFrame({
    'key1':['A','A','B','B','A'],
    'key2':['one','two','one','two','one'],
    'data1':[1,2,3,4,5],
    'data2':[10,20,30,40,50]
})
df2

,key1,key2,data1,data2
0,A,one,1,10
1,A,two,2,20
2,B,one,3,30
3,B,two,4,40
4,A,one,5,50


In [15]:
groups = df2.groupby(df2.key1)
groups

In [16]:
groups.groups

{'A': [0, 1, 4], 'B': [2, 3]}

In [18]:
groups.groups.keys()

dict_keys(['A', 'B'])

In [19]:
groups.groups['A']

Int64Index([0, 1, 4], dtype='int64')

In [20]:
pd.DataFrame(groups)

,0,1
0,A,key1 key2 data1 data2 0 A one 1 ...
1,B,key1 key2 data1 data2 2 B one 3 ...


In [21]:
pd.DataFrame(groups).loc[0].values

array(['A',   key1 key2  data1  data2
            0    A  one      1     10
            1    A  two      2     20
            4    A  one      5     50], dtype=object)

In [22]:
pd.DataFrame(groups).loc[1].values

array(['B',   key1 key2  data1  data2
            2    B  one      3     30
            3    B  two      4     40], dtype=object)

In [23]:
pd.DataFrame(groups).loc[1].values[1]

,key1,key2,data1,data2
2,B,one,3,30
3,B,two,4,40


In [24]:
groups.sum()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_18700\1171057664.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  groups.sum()


,data1,data2
key1,,
A,8,80
B,7,70


In [25]:
groups['data1'].sum()

key1
A    8
B    7
Name: data1, dtype: int64

In [26]:
groups.sum()['data1']

C:\Users\Administrator\AppData\Local\Temp\ipykernel_18700\1934223636.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  groups.sum()['data1']


key1
A    8
B    7
Name: data1, dtype: int64

In [27]:
groups[['data1']].sum()
groups[['data2']].sum()

,data1
key1,
A,8
B,7


,data2
key1,
A,80
B,70


#### 그룹 함수 예제
- `apply()/agg()`
    - df 등에 벡터라이징 연산을 적용(`axis=0/1`)
    - `agg` 함수는 숫자 타입의 스칼라만 리턴

In [28]:
import seaborn as sns
iris = sns.load_dataset('iris')

In [29]:
iris

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [30]:
i_groups = iris.groupby(iris.species)

In [31]:
i_groups

In [32]:
i_groups.mean()

,sepal_length,sepal_width,petal_length,petal_width
species,,,,
setosa,5.006,3.428,1.462,0.246
versicolor,5.936,2.770,4.260,1.326
virginica,6.588,2.974,5.552,2.026


In [34]:
pd.DataFrame(i_groups)

,0,1
0,setosa,sepal_length sepal_width petal_length p...
1,versicolor,sepal_length sepal_width petal_length p...
2,virginica,sepal_length sepal_width petal_length ...


In [35]:
pd.DataFrame(i_groups).loc[1]

0                                           versicolor
1        sepal_length  sepal_width  petal_length  p...
Name: 1, dtype: object

In [37]:
pd.DataFrame(i_groups).loc[1].values[1].head(5)

,sepal_length,sepal_width,petal_length,petal_width,species
50,7.0,3.2,4.7,1.4,versicolor
51,6.4,3.2,4.5,1.5,versicolor
52,6.9,3.1,4.9,1.5,versicolor
53,5.5,2.3,4.0,1.3,versicolor
54,6.5,2.8,4.6,1.5,versicolor


In [33]:
pd.DataFrame(i_groups).loc[1].values[1]['sepal_width'].max()

3.4

In [38]:
i_groups.petal_length.sum()

species
setosa         73.1
versicolor    213.0
virginica     277.6
Name: petal_length, dtype: float64

In [39]:
i_groups.max()/i_groups.min()

,sepal_length,sepal_width,petal_length,petal_width
species,,,,
setosa,1.348837,1.913043,1.900000,6.000000
versicolor,1.428571,1.700000,1.700000,1.800000
virginica,1.612245,1.727273,1.533333,1.785714


In [40]:
def peak_to_peak_ratio(x):
    return x.max() / x.min()
peak_to_peak_ratio(iris.sepal_length)

1.8372093023255816

- `agg()`, `apply()` 예제

In [41]:
# 그룹 객체의 각 그룹에 대해 열별로 함수 호출
i_groups.agg(peak_to_peak_ratio)

,sepal_length,sepal_width,petal_length,petal_width
species,,,,
setosa,1.348837,1.913043,1.900000,6.000000
versicolor,1.428571,1.700000,1.700000,1.800000
virginica,1.612245,1.727273,1.533333,1.785714


In [42]:
i_groups.apply(peak_to_peak_ratio)

,sepal_length,sepal_width,petal_length,petal_width
species,,,,
setosa,1.348837,1.913043,1.900000,6.000000
versicolor,1.428571,1.700000,1.700000,1.800000
virginica,1.612245,1.727273,1.533333,1.785714


- `apply()` 예제

In [43]:
def top3_petal_length(df):
    return df.sort_values(by='petal_length', ascending=False)[:3]
top3_petal_length(iris)

,sepal_length,sepal_width,petal_length,petal_width,species
118,7.7,2.6,6.9,2.3,virginica
122,7.7,2.8,6.7,2.0,virginica
117,7.7,3.8,6.7,2.2,virginica


In [44]:
i_groups.apply(top3_petal_length)

sepal_length  sepal_width  petal_length  petal_width  \
species                                                                
setosa     24            4.8          3.4           1.9          0.2   
           44            5.1          3.8           1.9          0.4   
           23            5.1          3.3           1.7          0.5   
versicolor 83            6.0          2.7           5.1          1.6   
           77            6.7          3.0           5.0          1.7   
           72            6.3          2.5           4.9          1.5   
virginica  118           7.7          2.6           6.9          2.3   
           117           7.7          3.8           6.7          2.2   
           122           7.7          2.8           6.7          2.0   

                   species  
species                     
setosa     24       setosa  
           44       setosa  
           23       setosa  
versicolor 83   versicolor  
           77   versicolor  
           72   versicolor  
virginica  118   virginica  
           117   virginica  
           122   virginica

In [46]:
# i_groups.agg(top3_petal_length)
# agg에 적용된 함수의 반환값이 수치 스칼라가 아니기 때문에

- `apply()` 예제2

In [52]:
def q3cut(s):
    return pd.qcut(s, 3, labels=['소','중','대']).astype(str)

In [53]:
i_groups.petal_length.apply(q3cut)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_18700\1756483873.py:1: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  i_groups.petal_length.apply(q3cut)


0      소
1      소
2      소
3      중
4      소
      ..
145    소
146    소
147    소
148    중
149    소
Name: petal_length, Length: 150, dtype: object

In [54]:
q3cut(iris.petal_length)

0      소
1      소
2      소
3      소
4      소
      ..
145    대
146    대
147    대
148    대
149    대
Name: petal_length, Length: 150, dtype: object

In [55]:
iris.head(5)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [57]:
iris['petal_length_class'] = iris.groupby(iris.species).petal_length.apply(q3cut)
iris.head()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_18700\1629056941.py:1: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  iris['petal_length_class'] = iris.groupby(iris.species).petal_length.apply(q3cut)


,sepal_length,sepal_width,petal_length,petal_width,species,petal_length_class
0,5.1,3.5,1.4,0.2,setosa,소
1,4.9,3.0,1.4,0.2,setosa,소
2,4.7,3.2,1.3,0.2,setosa,소
3,4.6,3.1,1.5,0.2,setosa,중
4,5.0,3.6,1.4,0.2,setosa,소


### 그룹함수 및 피봇테이블 이용 간단한 분석 예제

#### 식당에서 식사 후 내는 팁과 관련된 데이터 이용
- seaborn 패키지 내 tips 데이터셋 사용